# Faruq-v3 ACMC — single locked test

Tahap final tanpa training. Notebook membuka test asli Faruq satu kali, membuang parent/hash yang pernah muncul di development, menghapus pseudo-replikasi parent, menjalankan gate kelayakan, lalu hanya jika PASS mengevaluasi enam checkpoint beku D0FT/ACMC1. Tidak ada tuning setelah test dibuka.

In [ ]:
from google.colab import drive, userdata
drive.mount('/content/drive')

import json, os, shutil, subprocess, sys, tarfile, time
from pathlib import Path

REPO = Path('/content/coffee-bean-detection')
BRANCH = 'agent/add-vadcp-pipeline'
os.chdir('/content')
if REPO.exists():
    shutil.rmtree(REPO)
clone = ['git', 'clone', '--depth', '1', '--branch', BRANCH, 'https://github.com/ediprin/coffee-bean-detection.git', str(REPO)]
for attempt in range(1, 4):
    result = subprocess.run(clone)
    if result.returncode == 0:
        break
    if REPO.exists():
        shutil.rmtree(REPO)
    if attempt == 3:
        raise RuntimeError('Git clone gagal tiga kali.')
    time.sleep(2)
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-e', str(REPO), 'roboflow'], check=True)
for module_name in list(sys.modules):
    if module_name == 'coffee_detector' or module_name.startswith('coffee_detector.'):
        sys.modules.pop(module_name, None)
sys.path.insert(0, str(REPO / 'src'))
os.chdir(REPO)
print('SETUP SELESAI:', REPO)

In [ ]:
from coffee_detector.drive_project import require_project_artifact, resolve_drive_project_root

REQUIRED = (
    'bundles/faruq-development-v3-grouped.tar',
    'experiments/faruq-v3-acmc-paired-confirmation-v1/val_reports/acmc1_paired_optimization_confirmation.json',
    'experiments/faruq-v3-acmc-optimization-control-v1/D0FT_seed42/weights/best.pt',
    'experiments/faruq-v3-acmc-one-stage-v1/ACMC1_seed42/weights/best.pt',
    'experiments/faruq-v3-acmc-paired-confirmation-v1/D0FT/D0FT_seed123/weights/best.pt',
    'experiments/faruq-v3-acmc-paired-confirmation-v1/ACMC1/ACMC1_seed123/weights/best.pt',
    'experiments/faruq-v3-acmc-paired-confirmation-v1/D0FT/D0FT_seed2026/weights/best.pt',
    'experiments/faruq-v3-acmc-paired-confirmation-v1/ACMC1/ACMC1_seed2026/weights/best.pt',
)
PROJECT_ROOT = resolve_drive_project_root(required_relative_paths=REQUIRED)
DEV_ARCHIVE = require_project_artifact(PROJECT_ROOT, REQUIRED[0])
CONFIRMATION = require_project_artifact(PROJECT_ROOT, REQUIRED[1])
D0FT = tuple(require_project_artifact(PROJECT_ROOT, path) for path in (REQUIRED[2], REQUIRED[4], REQUIRED[6]))
ACMC = tuple(require_project_artifact(PROJECT_ROOT, path) for path in (REQUIRED[3], REQUIRED[5], REQUIRED[7]))
DEV_ROOT = Path('/content/faruq-development-v3-grouped')
RAW_ROOT = Path('/content/faruq-segmentation-raw')
LOCKED_ROOT = Path('/content/faruq-v3-locked-test')
EVIDENCE_ROOT = PROJECT_ROOT / 'evidence/faruq-v3-locked-test-v1'
LOCKED_ARCHIVE = PROJECT_ROOT / 'bundles/faruq-v3-locked-test-v1.tar'
FINAL_ROOT = PROJECT_ROOT / 'experiments/faruq-v3-acmc-locked-test-v1'
EVIDENCE_ROOT.mkdir(parents=True, exist_ok=True)
LOCKED_ARCHIVE.parent.mkdir(parents=True, exist_ok=True)
FINAL_ROOT.mkdir(parents=True, exist_ok=True)
if not (DEV_ROOT / 'faruq_grouped_manifest.json').is_file():
    with tarfile.open(DEV_ARCHIVE, 'r') as archive:
        archive.extractall('/content', filter='data')
print('PROJECT:', PROJECT_ROOT)
print('DEV    :', DEV_ROOT)
print('FINAL  :', FINAL_ROOT)

In [ ]:
# Audit test hanya dilakukan sekali. Jika paket Drive sudah ada, restore tanpa membuka Roboflow lagi.
if not (LOCKED_ROOT / 'faruq_locked_test_eligibility.json').is_file() and LOCKED_ARCHIVE.is_file():
    with tarfile.open(LOCKED_ARCHIVE, 'r') as archive:
        archive.extractall('/content', filter='data')

if not (LOCKED_ROOT / 'faruq_locked_test_eligibility.json').is_file():
    from roboflow import Roboflow
    api_key = userdata.get('ROBOFLOW_API_KEY')
    assert api_key, 'Tambahkan Colab secret ROBOFLOW_API_KEY.'
    rf = Roboflow(api_key=api_key)
    downloaded = rf.workspace('situju-kamkape').project('robusta_sni_dataset-hr9ci').version(1).download('coco-segmentation', location=str(RAW_ROOT))
    RAW_ROOT = Path(downloaded.location)
    if LOCKED_ROOT.exists():
        shutil.rmtree(LOCKED_ROOT)
    from coffee_detector.prepare_faruq_locked_test import prepare_faruq_locked_test
    eligibility = prepare_faruq_locked_test(RAW_ROOT, DEV_ROOT / 'faruq_grouped_manifest.json', LOCKED_ROOT)
    temporary = Path('/content/faruq-v3-locked-test-v1.tar')
    with tarfile.open(temporary, 'w') as archive:
        archive.add(LOCKED_ROOT, arcname='faruq-v3-locked-test')
    shutil.copy2(temporary, LOCKED_ARCHIVE)
    temporary.unlink()
else:
    eligibility = json.loads((LOCKED_ROOT / 'faruq_locked_test_eligibility.json').read_text())
for name in ('faruq_locked_test_eligibility.json', 'faruq_locked_test_manifest.json', 'faruq_locked_test_excluded.json', 'faruq_locked_test_quarantine.json'):
    source = LOCKED_ROOT / name
    if source.is_file():
        shutil.copy2(source, EVIDENCE_ROOT / name)
print(json.dumps(eligibility, indent=2, ensure_ascii=False))

In [ ]:
import pandas as pd
from IPython.display import display

class_table = pd.DataFrame({
    'class_name': list(eligibility['instances_by_class']),
    'instances': list(eligibility['instances_by_class'].values()),
    'independent_parents': [eligibility['parents_by_class'][name] for name in eligibility['instances_by_class']],
})
display(class_table)
print('DECISION:', eligibility['decision'])
print('GATES   :', eligibility['gates'])
print('TEST IMAGES:', eligibility['materialized_images'])
if eligibility['decision'] != 'PASS':
    raise RuntimeError('STOP: test independen tidak cukup kuat. Tidak ada model yang diinference.')
print('PASS: locked test layak; enam checkpoint beku akan dievaluasi tepat sekali.')

In [ ]:
import torch
assert torch.cuda.is_available(), 'Aktifkan T4 GPU untuk inference final.'
command = [
    sys.executable, '-u', '-m', 'coffee_detector.experiments.run_faruq_v3_acmc_locked_test',
    '--test-root', str(LOCKED_ROOT),
    '--eligibility-summary', str(LOCKED_ROOT / 'faruq_locked_test_eligibility.json'),
    '--confirmation-summary', str(CONFIRMATION),
    '--output-root', str(FINAL_ROOT),
    '--d0ft-checkpoints', *map(str, D0FT),
    '--acmc-checkpoints', *map(str, ACMC),
    '--seeds', '42', '123', '2026',
    '--device', '0', '--authorize-test',
]
print('MENJALANKAN:', ' '.join(command), flush=True)
subprocess.run(command, cwd=REPO, check=True)

In [ ]:
SUMMARY = FINAL_ROOT / 'faruq_v3_acmc_locked_test_summary.json'
assert SUMMARY.is_file(), SUMMARY
final = json.loads(SUMMARY.read_text())
rows = []
for metric, values in final['aggregate'].items():
    rows.append({'metric': metric, **values})
display(pd.DataFrame(rows).style.format({key: '{:.2%}' for key in ('d0ft_mean', 'd0ft_std', 'acmc1_mean', 'acmc1_std', 'head_delta_mean', 'head_delta_std', 'head_delta_min')}))
print('CONCLUSION:', final['conclusion'])
print('CRITERIA  :', final['criteria'])
print('TRAINING  :', final['training_executed'])
print('NEXT      :', final['next_action'])
print('SUMMARY   :', SUMMARY)
print('Kirim tabel dan conclusion. Test sudah dibuka; jangan tuning atau training lagi.')